# Frame 1 (Day -2) Morphology vs. Eventual Fate Mixture - vem 1uM, 48hrs Pre-Treatment

Looks only at frame 1 (2 days before the drug is added) and asks: does a
cell's *shape* at that early timepoint relate to what its lineage will
eventually become by day 0?

Unlike `01_subclass_by_frame.ipynb` and `02_subclass_separation_stats.ipynb`,
frame-1 cells don't have a single subclass label - `cloneClassification`
only classifies cells present at day 0 (frame 3), and by then a frame-1
lineage can have already split into several day-0 descendants with
*different* fates. So each frame-1 cell gets a **mixture**: the fraction
of its day-0 descendants that end up dying / persisting / slow
proliferator / fast proliferator, from `lineage_frame3_to_frame1.mat` and
`clone_kmeans_id.mat`.

This notebook is purely visual (no statistical test) - a weighted KDE per
fate category, where each frame-1 cell contributes to every category's
contour in proportion to its own percentage in that category.

Requires `main.py` to have already been run (for `selected_metadata.csv`
and `{cell,nuc}_qgw_075P_dists.csv` under `output/cajal_output/`), and
`config.RAW_DRUG_DIR` / `LINEAGE_MAT_PATH` / `CLONE_KMEANS_MAT_PATH` to
point at the real raw data.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import umap
import matplotlib.pyplot as plt

# assumes Jupyter's cwd is this notebook's own directory
# (analyses/vem_48hr_pretreatment/notebooks/), so the analysis dir - and
# config.py - is one level up, and the repo root - where shared/ lives -
# is two more levels up from there
NOTEBOOK_DIR = os.getcwd()
ANALYSIS_DIR = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
REPO_ROOT    = os.path.abspath(os.path.join(ANALYSIS_DIR, '..', '..'))
sys.path.insert(0, REPO_ROOT)
sys.path.insert(0, ANALYSIS_DIR)
import config

# config.py's OUTPUT_DIR/CAJAL_OUTPUT_DIR are relative to the analysis
# directory, not this notebook's own cwd - re-anchor them to an absolute
# path once, here
config.OUTPUT_DIR       = os.path.join(ANALYSIS_DIR, config.OUTPUT_DIR)
config.CAJAL_OUTPUT_DIR = os.path.join(ANALYSIS_DIR, config.CAJAL_OUTPUT_DIR)

from shared.cajal.embed import load_distance_matrix
from shared.load_data import match_labels_to_metadata
from shared.density_plots import plot_umap_with_weighted_kde
import lineage_fate

## Method

For each modality (cell, nucleus):

1. **Load the already-computed distance matrix** for all 3 frames
   together (`load_distance_matrix`) - reused, not recomputed, since the
   GW distance between two frame-1 cells doesn't depend on what else was
   in that CAJAL run.
2. **Restrict to frame 1** - subset the distance matrix down to just the
   frame-1 × frame-1 block.
3. **Fit a fresh UMAP** on just that frame-1 submatrix (`metric='precomputed'`)
   - a frame-1-only layout, not a slice of the full 3-frame embedding, so
   the 2D positions reflect only frame-1 cells' relationships to each
   other. This is for visualization only, same as every other UMAP use
   in this project.
4. **Resolve each frame-1 lineage's fate mixture**:
   - `lineage_fate.build_raw_to_global_lin_id` resolves
     `lineage_frame3_to_frame1.mat`'s track-local
     `(file_idx, old_clone_id_frame1)` IDs into the same global `lin_id`
     already in `{drug}_metadata.csv`, by directly reusing
     process_imaging_data's own `load_track`/`make_master_ids`.
   - `lineage_fate.compute_frame1_fate_percentages` then gives each
     frame-1 lineage's `pct_dying`/`pct_persisting`/`pct_slow_prolif`/`pct_fast_prolif`.
   - A frame-1 cell whose lineage dies out before reaching day 0 has no
     entry here (no day-0 descendants to classify) - it gets 0% in every
     category, i.e. it contributes to none of the four contours, rather
     than being dropped from the plot entirely.
5. **Plot**: one weighted KDE contour per fate category over the frame-1
   UMAP, background scatter in plain grey (`plot_umap_with_weighted_kde`,
   levels 0.5/0.7/0.9).

In [ ]:
SUBCLASS_COLORS = {
    'dying':       'tab:red',
    'persisting':  'tab:purple',
    'slow_prolif': 'tab:orange',
    'fast_prolif': 'tab:green',
}
WEIGHT_COLS = {
    'dying':       'pct_dying',
    'persisting':  'pct_persisting',
    'slow_prolif': 'pct_slow_prolif',
    'fast_prolif': 'pct_fast_prolif',
}
PLOTS_DIR = os.path.join(config.OUTPUT_DIR, 'plots', '03_frame1_fate_mixture')
os.makedirs(os.path.join(PLOTS_DIR, 'pdfs'), exist_ok=True)
os.makedirs(os.path.join(PLOTS_DIR, 'svgs'), exist_ok=True)


def process_modality(modality, raw_to_global, fate_pct):
    '''
    Build the frame-1-only UMAP, attach each frame-1 cell's fate-mixture
    percentages, and plot the weighted KDE.

    Inputs:
        modality:      'cell' or 'nuc'
        raw_to_global: from lineage_fate.build_raw_to_global_lin_id
        fate_pct:      from lineage_fate.compute_frame1_fate_percentages
    Outputs:
        df: frame-1 cells' metadata + umap_0/umap_1 + pct_* columns (in-memory only)
    '''
    dist_csv = os.path.join(config.CAJAL_OUTPUT_DIR, f'{modality}_qgw_{config.N_POINTS:03d}P_dists.csv')
    selected_path = os.path.join(config.CAJAL_OUTPUT_DIR, 'selected_metadata.csv')
    selected = pd.read_csv(selected_path)

    labels, dist_matrix = load_distance_matrix(dist_csv)
    meta = match_labels_to_metadata(labels, selected)

    frame1_mask = (meta['frame'] == 1).to_numpy()
    frame1_dist = dist_matrix[np.ix_(frame1_mask, frame1_mask)]
    frame1_meta = meta.loc[frame1_mask].reset_index(drop=True)
    print(f"[{modality}] {len(frame1_meta)} frame-1 cells")

    reducer  = umap.UMAP(n_components=2, metric='precomputed', random_state=config.UMAP_SEED)
    frame1_embedding = reducer.fit_transform(frame1_dist)
    frame1_meta['umap_0'] = frame1_embedding[:, 0]
    frame1_meta['umap_1'] = frame1_embedding[:, 1]

    df = frame1_meta.merge(fate_pct, on='lin_id', how='left')
    pct_cols = list(WEIGHT_COLS.values())
    n_unmatched = df[pct_cols[0]].isna().sum()
    if n_unmatched > 0:
        print(f"  {n_unmatched}/{len(df)} frame-1 cell(s) have no day-0 descendants in "
              f"lineage_frame3_to_frame1.mat (lineage died before day 0) - 0% in every category")
    df[pct_cols] = df[pct_cols].fillna(0.0)

    fig, ax = plt.subplots(figsize=(6, 6))
    plot_umap_with_weighted_kde(
        df, WEIGHT_COLS, SUBCLASS_COLORS, ax,
        f'{modality} - frame 1 (day -2) fate mixture',
    )
    fig.savefig(os.path.join(PLOTS_DIR, 'pdfs', f'{modality}_frame1_fate_mixture.pdf'), bbox_inches='tight')
    fig.savefig(os.path.join(PLOTS_DIR, 'svgs', f'{modality}_frame1_fate_mixture.svg'), bbox_inches='tight')
    plt.close(fig)
    print(f"  Saved to {PLOTS_DIR}")

    return df


raw_to_global = lineage_fate.build_raw_to_global_lin_id(config.RAW_DRUG_DIR)
fate_pct = lineage_fate.compute_frame1_fate_percentages(
    config.LINEAGE_MAT_PATH, config.CLONE_KMEANS_MAT_PATH, raw_to_global,
)
print(f"{len(fate_pct)} frame-1 lineages have at least one day-0 descendant")

## Run: cell morphology

In [ ]:
cell_df = process_modality('cell', raw_to_global, fate_pct)
cell_df.head()

## Run: nuclear morphology

In [ ]:
nuc_df = process_modality('nuc', raw_to_global, fate_pct)
nuc_df.head()